# LittleBit Qwen3-8B on a free NVIDIA GPU

Choose a free GPU runtime and run all cells. No Drive mount or Hugging Face account is required. This benchmark keeps Qwen3 on CUDA and caches the unpacked factor signs as int8 in VRAM. Each factor multiplication uses FP32. The Steam Deck benchmark keeps factors packed and uses BF16 activations, so this compares complete inference setups. The pinned checkpoint, chat prompt, greedy decoding and 16-token limit match the Deck test. The download is about 3 GB. Allow at least 12 GiB of free GPU memory for this implementation.

In [ ]:
import subprocess, sys, torch
from pathlib import Path

if not torch.cuda.is_available():
    raise RuntimeError("Select Runtime > Change runtime type > GPU and reconnect.")
free, total = torch.cuda.mem_get_info()
print(torch.cuda.get_device_name(), f"{free/2**30:.2f} GiB free of {total/2**30:.2f}")
if free < 12 * 2**30:
    raise RuntimeError("This FP32 benchmark needs at least 12 GiB free VRAM.")
work = Path("/content/littlebit-benchmark")
work.mkdir(exist_ok=True)
subprocess.run([sys.executable, "-m", "venv", "--system-site-packages", str(work/"env")], check=True)
python = str(work/"env/bin/python")
subprocess.run([python, "-m", "pip", "install", "-q", "transformers==4.51.3", "accelerate==1.6.0", "huggingface-hub==0.30.2", "safetensors>=0.5.3"], check=True)
import urllib.request, hashlib
url = "https://raw.githubusercontent.com/ILayZ/littlebit-steamdeck/b6a22b8d9113358f7a9f0b23fcb61c292473e0b4/deck_qwen.py"
source = urllib.request.urlopen(url).read()
assert hashlib.sha1(b"blob " + str(len(source)).encode() + b"\0" + source).hexdigest() == "bf8e947b197bd80172a68f2e84c631375eef4414"
(work/"deck_qwen.py").write_bytes(source)

In [ ]:
program = r'''
import json, statistics, time
from pathlib import Path
import numpy as np
import torch
from huggingface_hub import snapshot_download
from transformers import AutoTokenizer
from deck_qwen import MODEL, REV, load, signs, benchmark_prompt

class CudaFactors:
    def linear(self, x, weight, shape):
        return torch.nn.functional.linear(x, weight.float())

def run():
    path = Path(snapshot_download(MODEL, revision=REV,
                allow_patterns=["*.json","*.safetensors","merges.txt","vocab.txt"]))
    tokenizer = AutoTokenizer.from_pretrained(path, local_files_only=True)
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.cuda.reset_peak_memory_stats()
    start = time.perf_counter()
    model = load(path, CudaFactors())
    for layer in model.modules():
        if hasattr(layer, "parts"):
            converted = []
            for v, vs, u, us, col, latent, row in layer.parts:
                def weight(packed, shape):
                    return torch.from_numpy(signs(packed.numpy(), shape)).to("cuda")
                converted.append((weight(v, vs), vs, weight(u, us), us,
                                  col.to("cuda", torch.float32), latent.to("cuda", torch.float32),
                                  row.to("cuda", torch.float32)))
            layer.parts = converted
    model.to(device="cuda", dtype=torch.float32).eval()
    ids = benchmark_prompt(tokenizer).to("cuda")
    torch.cuda.synchronize()
    load_s = time.perf_counter() - start
    runs = []
    for index in range(3):
        start = time.perf_counter()
        out = model(ids, use_cache=True, logits_to_keep=1)
        token = out.logits[:, -1].argmax(-1, keepdim=True)
        generated = [token.item()]
        first = time.perf_counter() - start
        start = time.perf_counter()
        while len(generated) < 16 and generated[-1] != tokenizer.eos_token_id:
            out = model(token, past_key_values=out.past_key_values, use_cache=True, logits_to_keep=1)
            token = out.logits[:, -1].argmax(-1, keepdim=True)
            generated.append(token.item())
        elapsed = time.perf_counter()-start
        runs.append(dict(first_token_s=first, output_tokens=len(generated),
                         decode_tokens_s=(len(generated)-1)/elapsed if len(generated)>1 else None,
                         text=tokenizer.decode(generated, skip_special_tokens=True)))
        del out
    rates = [r["decode_tokens_s"] for r in runs if r["decode_tokens_s"] is not None]
    report = dict(model=MODEL, revision=REV, device=torch.cuda.get_device_name(),
                  torch=torch.__version__, cuda=torch.version.cuda, prompt_tokens=ids.shape[1],
                  load_s=load_s, peak_allocated_vram_mib=torch.cuda.max_memory_allocated()/2**20,
                  runs=runs, median_decode_tokens_s=statistics.median(rates) if rates else None)
    print(json.dumps(report, indent=2))
    Path("/content/littlebit-colab.json").write_text(json.dumps(report, indent=2))

with torch.inference_mode():
    run()
'''
(work/"run.py").write_text(program)
subprocess.run([python, "-u", "run.py"], cwd=work, check=True)

The JSON report is also available in the Files panel at `/content/littlebit-colab.json`. Peak allocated VRAM is a PyTorch allocator metric. The Deck logs report process RSS.